## ANN Baymax

In [16]:
import pandas as pd
import random 

df = pd.read_csv("Training.csv").copy()
print("Dup:", df.duplicated().sum())

unique = df.drop_duplicates().copy()
duplicated = df[df.duplicated()].copy()



def mutate(duplicated):
    mutation_rate = 0.15
    columns = duplicated.columns[:-1] # removing prognosis
    
    for i in range(len(duplicated)):
        
        for j in range(len(columns)):
            
            if random.random() < mutation_rate:
                
                duplicated.iat[i ,j] = 1 - duplicated.iat[i ,j]
                
    return duplicated


mutated_df = mutate(duplicated)
print("After mutation duplicates:", mutated_df.duplicated().sum())
    
final_dataset = pd.concat([unique, mutated_df], ignore_index=True)

print("final dataset len:", len(final_dataset))
print("final dataset duplicates:", final_dataset.duplicated().sum())

Dup: 4616
After mutation duplicates: 0
final dataset len: 4920
final dataset duplicates: 0


The original data has 4616 duplicated rows. After mutation, we introduced noise into the chromosomes which significantly decreases the duplication.


In [17]:
## Now coverting final dataset into training and testing components.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


X = final_dataset.drop(columns=['prognosis'])
print(X.columns)

Y = final_dataset['prognosis']
encoder = LabelEncoder()
Y = encoder.fit_transform(Y)


X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

print(f"\nX_train shape: {X_train.shape} (Training on {X_train.shape[0]} patients, {X_train.shape[1]} symptoms each)")
print(f"y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape} (test set of {X_test.shape[0]} patients)")
print(f"y_test shape: {Y_train.shape}")


Index(['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing',
       'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity',
       'ulcers_on_tongue',
       ...
       'pus_filled_pimples', 'blackheads', 'scurring', 'skin_peeling',
       'silver_like_dusting', 'small_dents_in_nails', 'inflammatory_nails',
       'blister', 'red_sore_around_nose', 'yellow_crust_ooze'],
      dtype='str', length=132)

X_train shape: (3936, 132) (Training on 3936 patients, 132 symptoms each)
y_train shape: (3936,)
X_test shape: (984, 132) (test set of 984 patients)
y_test shape: (3936,)


In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

baymax_tf_model = Sequential()

baymax_tf_model.add(Dense(units=64, activation='relu', input_shape=(132,))) # input 132 (symptoms) first hidden layer 64 neurons

baymax_tf_model.add(Dropout(0.2)) # Kill 20% chromosomes for better learning 

baymax_tf_model.add(Dense(units=32, activation='relu')) #second hidden layer with 32 neurons

baymax_tf_model.add(Dense(units=41, activation='softmax')) # output layer with softmax acitvation

baymax_tf_model.summary()


baymax_tf_model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

history = baymax_tf_model.fit(
    X_train, 
    Y_train,
    epochs=50,            
    batch_size=32,        
    validation_data=(X_test,Y_test), 
    verbose=1             
)




/home/Ahsane/Desktop/AiLab/Lab Project/.venv313/lib64/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         8,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 41)             │         1,353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,945 (46.66 KB)

 Trainable params: 11,945 (46.66 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0744 - loss: 3.6149 - val_accuracy: 0.2165 - val_loss: 3.3688
Epoch 2/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2990 - loss: 2.8956 - val_accuracy: 0.5112 - val_loss: 2.2363
Epoch 3/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4936 - loss: 1.9991 - val_accuracy: 0.6189 - val_loss: 1.5800
Epoch 4/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5991 - loss: 1.5030 - val_accuracy: 0.6778 - val_loss: 1.2547
Epoch 5/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6588 - loss: 1.2533 - val_accuracy: 0.7104 - val_loss: 1.1126
Epoch 6/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6913 - loss: 1.0814 - val_accuracy: 0.7246 - val_loss: 1.0076
Epoch 7/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7327 - loss: 0.9654 - val_accuracy: 0.7459 - val_loss: 0.9199
Epoch 8/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7436 - loss: 0.8887 - val_accuracy: 0.

In [19]:
import numpy as np

print("--- Running Test Case 1: Manual Patient Simulation ---")
symptom_cols = final_dataset.columns[:-1]
# 1. Create a blank patient (132 zeros)
simulated_patient = np.zeros(len(symptom_cols))

# 2. Define the symptoms our dummy patient is experiencing
active_symptoms = ['chills', 'high_fever', 'sweating', 'muscle_pain']

for symptom in active_symptoms:
    if symptom in symptom_cols:
        i = list(symptom_cols).index(symptom)
        simulated_patient[i] = 1

input_data = simulated_patient.reshape(1, 132)

predictions = baymax_tf_model.predict(input_data, verbose=0)

winning_index = np.argmax(predictions)
confidence = np.max(predictions) * 100

predicted_disease = encoder.inverse_transform([winning_index])[0]

print(f"\nPatient Symptoms: {active_symptoms}")
print(f"Baymax Diagnosis: {predicted_disease}")
print(f"Confidence Level: {confidence:.2f}%")

# If Baymax outputs "Malaria" with >80% confidence, your model is brilliant!

--- Running Test Case 1: Manual Patient Simulation ---

Patient Symptoms: ['chills', 'high_fever', 'sweating', 'muscle_pain']
Baymax Diagnosis: Malaria
Confidence Level: 83.89%


In [21]:
import random

print("--- Running Test Case 2: Blindfold Verification ---")

# Pick 5 random patients from the test set
random_indices = random.sample(range(len(X_test)), 5)

for i, idx in enumerate(random_indices):
    # Grab the patient's symptoms
    patient_data = X_test.iloc[idx].values
    
    # Grab the true disease (the answer key)
    true_disease_int = Y_test[idx]
    true_disease_name = encoder.inverse_transform([true_disease_int])[0]
    
    # Ask Baymax to predict
    prediction_array = baymax_tf_model.predict(patient_data.reshape(1, 132), verbose=0)
    baymax_guess_int = np.argmax(prediction_array)
    baymax_guess_name = encoder.inverse_transform([baymax_guess_int])[0]
    confidence = np.max(prediction_array) * 100
    
    # Format the output
    match = "✅" if true_disease_name == baymax_guess_name else "❌"
    
    print(f"\nPatient {i+1}:")
    print(f"  Actual Disease:  {true_disease_name}")
    print(f"  Baymax Guess:    {baymax_guess_name} ({confidence:.1f}%) {match}")

--- Running Test Case 2: Blindfold Verification ---

Patient 1:
  Actual Disease:  Gastroenteritis
  Baymax Guess:    Gastroenteritis (97.5%) ✅

Patient 2:
  Actual Disease:  Varicose veins
  Baymax Guess:    Varicose veins (97.8%) ✅

Patient 3:
  Actual Disease:  Hepatitis D
  Baymax Guess:    Hepatitis D (97.5%) ✅

Patient 4:
  Actual Disease:  Gastroenteritis
  Baymax Guess:    Paralysis (brain hemorrhage) (62.2%) ❌

Patient 5:
  Actual Disease:  Bronchial Asthma
  Baymax Guess:    Bronchial Asthma (99.9%) ✅
